# Torch vs ONNX performance

In [2]:
import torch
import onnx
import onnxruntime
import lenet
import numpy as np

torch.manual_seed(42)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Test dataset

In [3]:
X = lenet.load_tensor(lenet.sample_filename)
y = lenet.load_tensor(lenet.labels_filename, dtype="<i8")

X.shape, y.shape

((64, 1, 28, 28), (64,))

## PyTorch model

In [4]:
lenet_model = lenet.LeNetModel().to(device)
lenet_model.load_state_dict(torch.load(lenet.torch_filename, map_location=device))
lenet_model.eval()


LeNetModel(
  (lenet_stack): Sequential(
    (0): Conv2d(1, 12, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1152, out_features=120, bias=True)
    (8): ReLU()
    (9): Linear(in_features=120, out_features=84, bias=True)
    (10): Sigmoid()
    (11): Linear(in_features=84, out_features=10, bias=True)
  )
)

## ONNX

In [5]:
onnx_model = onnx.load(lenet.onnx_filename)
onnx.checker.check_model(onnx_model)
print(f"onnx op_set version={onnx.defs.onnx_opset_version()}")
print(f"onnx_model.op_set_version={onnx_model.opset_import[0].version}")
print(f"onnx_model.ir_version={onnx_model.ir_version}")

model_input = onnx_model.graph.input[0]
print(f"onnx_model.input: name={model_input.name}, type={model_input.type.tensor_type.elem_type} shape={model_input.type.tensor_type.shape.dim}")

model_output = onnx_model.graph.output[0]
print(f"onnx_model.output: name={model_output.name}, type={model_output.type.tensor_type.elem_type} shape={model_output.type.tensor_type.shape.dim}")


onnx op_set version=28
onnx_model.op_set_version=20
onnx_model.ir_version=10
onnx_model.input: name=images, type=1 shape=[dim_value: 1
, dim_value: 1
, dim_value: 28
, dim_value: 28
]
onnx_model.output: name=logits, type=1 shape=[dim_value: 1
, dim_value: 10
]


## ONNX C++ inference

In [6]:
import subprocess
from pathlib import Path

project_root = Path(lenet.__file__).resolve().parent.parent
subprocess.run(
    [str(project_root / "build/onnx_exporter"), "model/lenet.onnx"],
    cwd=project_root,
    check=True,
)


ONNX Runtime v1.30.1
Available providers:
  CPUExecutionProvider
Running inference
[-3.205126,-1.3334465,-1.8047261,0.44999284,-6.5404344,1.4731627,-4.8560433,5.300681,-0.21304229,12.730886]
[3.1381612,-2.509291,11.851143,-1.0311589,4.314294,-5.834595,4.4311914,-9.748271,0.8965151,-4.657622]
[0.9101163,12.207221,-2.989013,2.6250296,0.76360136,-4.4869723,1.7526731,-3.9990015,-1.9131021,-2.8173313]
[2.6841667,13.510957,-0.9756792,4.2586074,0.05749645,-3.3802953,-0.25939542,-5.493281,-4.1746893,-4.9729633]
[9.532145,-3.592979,3.560283,1.5039285,-0.05797252,-2.114326,9.503459,-6.7855315,-0.93953,-8.531653]
[0.71737456,11.493952,-1.2516756,3.542276,1.1166223,-3.6392014,2.1093485,-5.3230085,-3.1626217,-4.0626535]
[0.7130701,-0.7304305,6.082156,-0.69082564,10.638499,-6.863804,6.009723,-8.594287,-2.3753371,-3.2120144]
[1.494726,-3.6408749,3.7526393,-0.65950584,6.115648,-1.1313285,10.830559,-9.171266,-2.1006484,-4.6012425]
[2.33658,-0.57904077,-1.1568441,-0.46890187,-9.585063,12.258955,-2.51194

CompletedProcess(args=['/home/arykovanov/src/learning/arm_preparation/build/onnx_exporter', 'model/lenet.onnx'], returncode=0)

In [7]:
cpp_logits = lenet.load_tensor(lenet.cpp_logits_filename)
cpp_logits.shape

(64, 10)

## Logit values

Run inference with both PyTorch and ONNX Runtime for all the presaved image batch

In [ ]:
onnxruntime_session = onnxruntime.InferenceSession(lenet.onnx_filename)

torch_logits = torch.zeros((len(X), 10))
onnx_logits = torch.zeros((len(X), 10))

with torch.no_grad():
    for idx in range(len(y)):
        img = X[idx]
        label = y[idx]
        pred_onnx = onnxruntime_session.run(None, {"images": img.reshape(1, *img.shape)})[0].reshape(-1)
        pred_torch = lenet_model(torch.from_numpy(img.reshape(1, 1, 28, 28)))
        torch_logits[idx] = pred_torch
        onnx_logits[idx] = torch.from_numpy(pred_onnx)


Check if ONNX and Pytorch prediction are the same.

In [9]:
print(f"label: {y[0]}")
print(f"torch: {torch_logits[0]}")
print(f"onnx: {onnx_logits[0]}")
print(f"cpp: {cpp_logits[0]}")

for idx in range(len(y)):
    label = y[idx]
    # print(np.abs(onnx_logits[idx] - torch_logits[idx]).max())
    if torch_logits[idx].argmax() != onnx_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_onnx={onnx_logits[idx].argmax()}")
    if torch_logits[idx].argmax() != cpp_logits[idx].argmax():
        print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={torch_logits[idx].argmax()}, pred_cpp={cpp_logits[idx].argmax()}")
    if abs(onnx_logits[idx] - torch_logits[idx]).max() > 1e-5:
        raise ValueError(f"Logits at sample {idx}: label={label}, pred_torch={torch_logits[idx]}, pred_onnx={onnx_logits[idx]}. The difference is {np.abs(onnx_logits[idx] - torch_logits[idx]).max()} and it is TOO BIG! The difference should be less than 1e-5.")


label: 9
torch: tensor([-3.2051, -1.3334, -1.8047,  0.4500, -6.5404,  1.4732, -4.8560,  5.3007,
        -0.2130, 12.7309])
onnx: tensor([-3.2051, -1.3334, -1.8047,  0.4500, -6.5404,  1.4732, -4.8560,  5.3007,
        -0.2130, 12.7309])
cpp: [-3.205126   -1.3334465  -1.8047261   0.44999284 -6.5404344   1.4731627
 -4.8560433   5.300681   -0.21304229 12.730886  ]
